# 보이스챗봇
1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다.
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.
(+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.

- 발표
    - 주제
    - 프롬프트 엔지니어링 핵심
    - 시연

In [1]:
# !pip install pyttsx3

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from openai import OpenAI 
import speech_recognition as sr 
import pyttsx3

client = OpenAI() 

In [11]:
# 1. 사용자 입력을 음성으로 받는다.
# 2. STT를 적용하여 텍스트로 변환한다. 
    
def listen():
    recognizer = sr.Recognizer()

    with sr.Microphone() as source:

        audio = recognizer.listen(source)
        txt = recognizer.recognize_google(audio, language='ko-KR')
        
        return txt

In [12]:
# 3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다. 

chat_history = []

def coach_reply(user_text, temperature=0.3):

    chat_history.append(
        {
            'role': 'user',
            'content': user_text
        }
    )

    system_instruction = """
    넌 이제부터 보디빌딩 대회에서 우승 수상이 여러 번 있는 유능하고 친근한 트레이너야.
    이제부터 사용자의 맞춤 운동 코치가 되는거야.

    ### 상황분석 ###
    - 사용자의 음성 텍스트를 분석
    - 입력된 대화에 기반해 운동 부위(상체, 하체, 유산소 등)를 파악
    - 먼저 운동 부위에 맞춰 스트레칭 안내
    - 입력된 대화에 기반한 운동 종류, 난이도(초급, 중급, 고급), 시간(분), 빈도(몇 세트, 몇 회), 운동 설명 안내
    - 입력된 대화에 기반한 사용자의 운동 완료 여부, 피로도, 기분 파악
    - 운동이 끝날 때마다 응원과 함께 동기부여 제시
    - 모든 운동이 끝났을 경우, 사용자에게 종료 안내 (오늘 운동은 여기까지야💪 맛있게 운동했다! 이제 종료라고 말하면 돼. 내일도 보자🖐️)

    ### 출력 형식 ###
    - 채팅과 같은 대화형 답변 (한국어, 친근한 말투, 1~2줄 이내)

    ### 예시 ###
    - 코치: 같이 운동해 보자. 오늘은 어떤 부위를 불태우고 싶어?
    - 사용자: 하체
    - 코치: 이야, 하체라니! 아주 제대로 마음먹었네. 오늘 한번 튼튼한 하체 만들러 가보자고. 난이도는 어느 정도로 해볼까? 초급, 중급, 고급 중에 말해주면 딱 맞춰서 짜줄게
    - 사용자: 오늘 좀 피곤해
    - 코치: 그럴 수 있지, 충분히 이해해. 그래도 조금이라도 움직이면 몸이 훨씬 가벼워질 거야. 오늘은 무리하지 않는 선에서 가볍게 시작해볼까?
    - 사용자: 좋아
    - 코치: 그럼 운동 전 스트레칭부터 해볼까? 가볍게 하체 위주로 할게. 먼저 다리 들어올리기 해볼게. 좌우 번갈아가며 2회 반복할거야. 서서 왼쪽 다리를 가슴 쪽으로 끌어당겨 15초 유지해줘.
    - 사용자: 했어
    - 코치: 잘했어. 다음 오른쪽도 15초 해보자
    - 사용자: 했어

"""

    stream_response = client.chat.completions.create(
        model='gpt-4o', 
        messages=[
            {
                'role': 'system', 
                'content': [
                    {
                        'type': 'text',
                        'text': system_instruction
                    }
                ] 
            }
        ] + chat_history, 
        response_format={
            'type' : 'text'
        }, 
        temperature=temperature,
        max_tokens=2048, 
        top_p=1, 
        frequency_penalty=1,
        presence_penalty=1,
        stream=True 
    )

    coach_text = ''
    for chunk in stream_response:
        content = chunk.choices[0].delta.content 

        if content is not None:
            coach_text += content

    chat_history.append(
        {
            'role': 'assistant',
            'content': coach_text
        }
    )

    return coach_text 

In [13]:
# 4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다. -- pyttsx

def text_to_speech(text, is_start=False):
    engine = pyttsx3.init()

    # 음성 속도 설정
    engine.setProperty('rate', 150)
    if is_start:
        text = '코치봇🤖: 자, 오늘 같이 한번 제대로 운동해 보자고. 혹시 오늘은 어떤 부위를 불태우고 싶어?'
        print(text)
        
    engine.say(text)
    engine.runAndWait()

In [14]:
# (+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.
def voice_talk():
    text_to_speech('', is_start=True)
    
    while True:
        user_text = listen()
        if user_text == '종료':
            print('맛있게 운동했다! 내일도 보자🖐️')
            break

        coach_text = coach_reply(user_text)

        print(f'사용자💪: {user_text}')
        print(f'코치봇🤖: {coach_text}')

        text_to_speech(coach_text)

In [8]:
voice_talk()

코치봇🤖: 자, 오늘 같이 한번 제대로 운동해 보자고. 혹시 오늘은 어떤 부위를 불태우고 싶어?


KeyboardInterrupt: 